# Degeneracy-Aware Sampling with Real Data

This notebook demonstrates degeneracy-aware sampling by:
1. Loading a real frame from the HO3D dataset (RGB, Depth, Mask).
2. Loading the current tracked 3D points from `meta_data.npz`.
3. Generating candidate 3D points from the segmentation mask using the depth map.
4. Selecting the best candidates to improve 3D covariance observability.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import imageio

# Add project root to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from point2pose.io.sources.dataset.datareader import Ho3dReader

%matplotlib inline

## 1. Data Loading Helper

Helper to load the tracked points from `meta_data.npz`.

In [ ]:
# Define data loading helper
def unpack_ragged(name: str, store: dict, dim=-1):
    data = store[f"{name}_data"]
    offsets = store[f"{name}_offsets"]
    lengths = store[f"{name}_lengths"]
    out = []
    for off, L in zip(offsets, lengths):
        flat_data = data[off : off + L]
        if dim == 3:
            reshaped_data = flat_data.reshape(-1, 3)
        elif dim == 2:
            reshaped_data = flat_data.reshape(-1, 2)
        else:
            reshaped_data = flat_data
        out.append(reshaped_data)
    return out

## 2. Sampler Implementation

The `DegeneracyAwareSampler` class implements the greedy selection strategy.

In [ ]:
class DegeneracyAwareSampler:
    def __init__(self, target_num_points=50):
        self.target_num_points = target_num_points

    def compute_eigen_score(self, points_3d):
        if len(points_3d) < 3:
            return 0.0, np.zeros(3)
        
        # Center the points
        mean = np.mean(points_3d, axis=0)
        centered = points_3d - mean
        
        # Compute covariance
        cov = (centered.T @ centered) / (len(points_3d) - 1)
        
        # Eigen decomposition
        eigvals, eigvecs = np.linalg.eigh(cov)
        
        # Sort eigenvalues and vectors (ascending)
        idx = eigvals.argsort()
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]
        
        # Objective: Maximize lambda_min / lambda_max (condition number inverse)
        # or just maximize lambda_min to expand in weak direction
        l1, l2, l3 = eigvals # l1 is min, l3 is max because eigh returns ascending
        
        min_lambda = np.maximum(eigvals[0], 1e-6)
        max_lambda = np.maximum(eigvals[2], 1e-6)
        
        ratio = min_lambda / max_lambda
        return ratio, eigvecs[:, 0] # Return smallest eigenvector

    def select_candidates(self, current_points, candidates, scores=None):
        """
        Greedy selection of candidates to improve 3D distribution.
        
        Args:
            current_points: (N, 3) array of existing 3D points
            candidates: (M, 3) array of candidate 3D points
            scores: (M,) array of base scores (quality * depth stability)
        """
        if len(candidates) == 0:
            return []
        
        if scores is None:
            scores = np.ones(len(candidates))
            
        selected_indices = []
        current_set = current_points.copy() if len(current_points) > 0 else np.zeros((0, 3))
        
        # Mask of available candidates
        available = np.ones(len(candidates), dtype=bool)
        
        # Calculate initial state
        current_ratio, weak_axis = self.compute_eigen_score(current_set)
        print(f"Initial set size: {len(current_set)}")
        print(f"Initial eigen-ratio: {current_ratio:.6f}")
        print(f"Initial weak axis: {weak_axis}")
        
        # Greedy loop
        num_to_select = min(self.target_num_points, len(candidates))
        
        for _ in range(num_to_select):
            best_idx = -1
            best_gain = -1.0
            
            candidate_indices = np.where(available)[0]
            
            # Strategy: Recompute eigen-ratio for each candidate
            for idx in candidate_indices:
                cand = candidates[idx]
                
                # Test set
                test_set = np.vstack([current_set, cand.reshape(1, 3)])
                new_ratio, _ = self.compute_eigen_score(test_set)
                
                gain = new_ratio - current_ratio
                
                # Combine with base score
                total_score = scores[idx] * (1.0 + gain * 10.0) # Weight gain heavily
                
                if total_score > best_gain:
                    best_gain = total_score
                    best_idx = idx
            
            if best_idx != -1:
                selected_indices.append(best_idx)
                available[best_idx] = False
                
                # Update current set
                current_set = np.vstack([current_set, candidates[best_idx].reshape(1, 3)])
                
                # Update metrics
                current_ratio, weak_axis = self.compute_eigen_score(current_set)
                # print(f"Selected {best_idx}, New Ratio: {current_ratio:.6f}")
            else:
                break
                
        return selected_indices

## 3. Load Data & Prepare Candidates

We use `Ho3dReader` to load the image/depth/mask and `meta_data.npz` for the tracked points.

In [ ]:
# Config
DATA_ROOT = "/home/justin/data/HO3D_V3/"
VIDEO_NAME = "MPM10"
META_DATA_PATH = f"/home/justin/code/point-to-pose/results/ho3d_single/{VIDEO_NAME}/meta_data/meta_data.npz"
FRAME_IDX = 260

# 1. Load Meta Data (Tracked Points)
if not os.path.exists(META_DATA_PATH):
    print(f"Error: Meta data not found at {META_DATA_PATH}")
    current_points_2d = None
else:
    print(f"Loading tracked points from {META_DATA_PATH}...")
    D = np.load(META_DATA_PATH, allow_pickle=True)
    track3d_list = unpack_ragged("track3d", D, dim=3)
    track2d_list = unpack_ragged("track2d", D, dim=2)
    visibles_list = unpack_ragged("visibles", D)
    valid_depth_list = unpack_ragged("valid_depth", D)
    
    if FRAME_IDX >= len(track3d_list):
        print(f"Frame {FRAME_IDX} out of bounds")
        current_points = np.zeros((0, 3))
        current_points_2d = np.zeros((0, 2))
    else:
        current_points = track3d_list[FRAME_IDX]
        current_points_2d = track2d_list[FRAME_IDX]
        visibles = visibles_list[FRAME_IDX].astype(bool)
        valid_depth = valid_depth_list[FRAME_IDX].astype(bool)
        
        # Filter by BOTH visibles AND valid_depth
        final_valid_mask = visibles
        
        # Also check for NaNs/Zeros in 3D points just in case
        finite_mask = np.all(np.isfinite(current_points), axis=1) & (np.linalg.norm(current_points, axis=1) > 0)
        
        final_mask = final_valid_mask & finite_mask
        
        current_points = current_points[final_mask]
        current_points_2d = current_points_2d[final_mask]
        print(f"Loaded {len(current_points)} tracked points for frame {FRAME_IDX} (Visible & Valid Depth)")

# 2. Load Frame Data (Candidates)
video_dir = os.path.join(DATA_ROOT, "evaluation", VIDEO_NAME)
if not os.path.exists(video_dir):
    print(f"Error: Video directory not found at {video_dir}")
else:
    print(f"Loading frame {FRAME_IDX} from {video_dir}...")
    reader = Ho3dReader(video_dir, DATA_ROOT)
    
    # Manually load image since get_color doesn't exist on Ho3dReader
    color_file = reader.color_files[FRAME_IDX]
    rgb = imageio.imread(color_file)
    
    # Get other data using existing methods
    mask = reader.get_mask(FRAME_IDX)
    xyz_map = reader.get_xyz_map(FRAME_IDX)
    
    # Visualize Mask
    plt.figure(figsize=(10, 4))
    plt.subplot(121)
    plt.imshow(rgb)
    plt.title("RGB")
    plt.subplot(122)
    plt.imshow(mask, cmap='gray')
    plt.title("Segmentation Mask")
    plt.show()
    
    # Generate Candidates from Mask
    # Mask contains 255 for object (or >0)
    mask_bool = (mask > 0)
    
    # Filter xyz_map by mask and validity (z > 0)
    # Note: depth2xyzmap returns 0 for invalid depth
    valid_depth_map = np.linalg.norm(xyz_map, axis=2) > 0.05
    
    final_cand_mask = mask_bool & valid_depth_map
    
    # Get the 3D points
    candidates = xyz_map[final_cand_mask]
    
    # IMPORTANT: We need the 2D pixel coordinates for projection later
    # Create meshgrid of indices
    H, W = xyz_map.shape[:2]
    ys, xs = np.indices((H, W))
    
    candidate_pixels = np.column_stack([xs[final_cand_mask], ys[final_cand_mask]])
    
    print(f"Found {len(candidates)} potential candidates in mask.")
    
    # Downsample candidates to avoid excessive computation
    if len(candidates) > 500:
        indices = np.random.choice(len(candidates), 500, replace=False)
        candidates = candidates[indices]
        candidate_pixels = candidate_pixels[indices]
        print(f"Downsampled to {len(candidates)} candidates.")

## 4. Run Sampler & Visualize

In [ ]:
if len(candidates) > 0 and len(current_points) > 0:
    # Run Sampler
    sampler = DegeneracyAwareSampler(target_num_points=30)
    selected_indices = sampler.select_candidates(current_points, candidates)
    
    selected_points = candidates[selected_indices]
    selected_pixels = candidate_pixels[selected_indices]
    
    # --- 2D Visualization ---
    plt.figure(figsize=(10, 8))
    plt.imshow(rgb)
    
    # Plot Existing Points (if 2D tracks are available)
    if current_points_2d is not None and len(current_points_2d) > 0:
        plt.scatter(current_points_2d[:, 0], current_points_2d[:, 1], c='blue', s=20, marker='o', alpha=0.6, label='Existing Points')
    
    # Plot Selected Candidates
    plt.scatter(selected_pixels[:, 0], selected_pixels[:, 1], c='red', s=40, marker='x', label='Selected Candidates')
    
    plt.title(f"Existing Points ({len(current_points)}) + Selected Candidates ({len(selected_points)})\nFiltered by Visibles & Valid Depth")
    plt.legend()
    plt.axis('off')
    plt.show()
    
    # --- 3D Visualization ---
    # Visualization with Plotly
    try:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        
        def get_ellipsoid_data(points, scale=2.0, n_points=20):
            if len(points) < 3: return None, None, None
            mean = np.mean(points, axis=0)
            cov = np.cov(points, rowvar=False)
            eigvals, eigvecs = np.linalg.eigh(cov)
            idx = eigvals.argsort()
            eigvals = eigvals[idx]
            eigvecs = eigvecs[:, idx]
            u = np.linspace(0, 2 * np.pi, n_points)
            v = np.linspace(0, np.pi, n_points)
            x = np.outer(np.cos(u), np.sin(v))
            y = np.outer(np.sin(u), np.sin(v))
            z = np.outer(np.ones_like(u), np.cos(v))
            radii = np.sqrt(np.maximum(eigvals, 0)) * scale
            for i in range(len(x)):
                for j in range(len(x)):
                    [x[i,j], y[i,j], z[i,j]] = mean + np.dot(eigvecs, np.array([x[i,j], y[i,j], z[i,j]]) * radii)
            return x, y, z
        
        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{'type': 'scene'}, {'type': 'xy'}]],
            subplot_titles=("Spatial Distribution & Covariance", "Covariance Eigenvalues"),
            column_widths=[0.6, 0.4]
        )
        
        # Existing points
        fig.add_trace(go.Scatter3d(
            x=current_points[:, 0], y=current_points[:, 1], z=current_points[:, 2],
            mode='markers', marker=dict(size=3, color='blue', opacity=0.3), name='Existing'
        ), row=1, col=1)
        
        # Initial Ellipsoid
        ex, ey, ez = get_ellipsoid_data(current_points, scale=2.5)
        if ex is not None:
            fig.add_trace(go.Surface(x=ex, y=ey, z=ez, opacity=0.15, showscale=False, colorscale='Blues', name='Init Cov'), row=1, col=1)
        
        # Candidates (Rejected)
        rejected_mask = np.ones(len(candidates), dtype=bool)
        rejected_mask[selected_indices] = False
        rejected_points = candidates[rejected_mask]
        fig.add_trace(go.Scatter3d(
            x=rejected_points[:, 0], y=rejected_points[:, 1], z=rejected_points[:, 2],
            mode='markers', marker=dict(size=2, color='red', opacity=0.1), name='Rejected'
        ), row=1, col=1)
        
        # Candidates (Selected)
        fig.add_trace(go.Scatter3d(
            x=selected_points[:, 0], y=selected_points[:, 1], z=selected_points[:, 2],
            mode='markers', marker=dict(size=7, color='gold', symbol='diamond', opacity=1.0, line=dict(color='black', width=1)), name='Selected'
        ), row=1, col=1)
        
        # Final Ellipsoid
        all_final = np.vstack([current_points, selected_points])
        fx, fy, fz = get_ellipsoid_data(all_final, scale=2.5)
        if fx is not None:
            fig.add_trace(go.Surface(x=fx, y=fy, z=fz, opacity=0.15, showscale=False, colorscale='Greens', name='Final Cov'), row=1, col=1)
        
        # Eigenvalues
        def get_eigs(pts):
            if len(pts) < 3: return [0,0,0]
            c = pts - np.mean(pts, axis=0)
            cov = (c.T @ c) / (len(pts)-1)
            return np.linalg.eigvalsh(cov)
            
        eigs_init = get_eigs(current_points)
        eigs_final = get_eigs(all_final)
        x_labels = ['Min (Weak)', 'Mid', 'Max']
        fig.add_trace(go.Bar(x=x_labels, y=eigs_init, name='Initial', marker_color='blue', opacity=0.7), row=1, col=2)
        fig.add_trace(go.Bar(x=x_labels, y=eigs_final, name='Final', marker_color='green', opacity=0.7), row=1, col=2)
        
        fig.update_layout(height=700, title_text="Sampling from Segmentation Mask Results")
        fig.show()
        
        print(f"Eigen-ratio: {eigs_init[0]/eigs_init[2]:.6f} -> {eigs_final[0]/eigs_final[2]:.6f}")
        
    except ImportError:
        print("Plotly not installed.")